# 概要
今月分のインセンティブシートを作成。

1.   プログラムを実施（Ctrl + F9）
2.   本ページの一番下にURLがスプレッドシートURLが表示



In [2]:
import datetime as dt
import os
import subprocess
import time

import google.auth
import gspread
import jwt
import requests
from google.colab import auth, userdata

auth.authenticate_user()

subprocess.run(
    ['pip', 'install', '-q', 'gspread', 'google-auth', 'google-api-python-client'],
    check=True,
)
subprocess.run(
    ['pip', 'install', '-q', 'pyjwt', 'cryptography', 'requests'],
    check=True,
)

os.environ['GITHUB_TOKEN'] = userdata.get('Github_token')  # トークンを設定

subprocess.run(
    [
        'pip', 'install', '-q',
        f'git+https://{os.environ["GITHUB_TOKEN"]}@github.com/rosca-tanakashi/rosca_common.git@main',
    ],
    check=True,
)

from rosca_common import get_employee_info

# ===== 設定 =====
# 対象年月（未入力で全期間）
TARGET_DATE = "2026-03-01"  # @param {type:"date"}
HEADER = ['氏名', '㊤決定粗利', '㊦決定粗利', '折半粗利合計']

SF_TOKEN_URL = 'https://login.salesforce.com/services/oauth2/token'
ZYUCYUBI_FIELD = 'zyucyubi__c'
UE_TANTO_FIELD = 'shinkikettei_uetanto__c'
SHITA_TANTO_FIELD = 'shinkikettei_shitatanto__c'
# =================


def is_target_date_set(value) -> bool:
    """日付が未入力なら False。"""
    if value is None:
        return False
    return str(value).strip() not in ('', 'None')


def to_target_date(value) -> dt.date:
    """Colab の date 入力値を date 型へ変換する。"""
    if isinstance(value, dt.datetime):
        return value.date()
    if isinstance(value, dt.date):
        return value
    return dt.datetime.strptime(str(value)[:10], '%Y-%m-%d').date()


def format_year_month_label(target_date_raw) -> str:
    """日付を「YYYY年MM月」形式へ変換する。"""
    target_date = to_target_date(target_date_raw)
    return f'{target_date.year}年{target_date.month:02d}月'


def resolve_output_year_month(target_date_raw) -> str:
    """スプレッドシート名・シート名用の年月ラベルを返す（未入力時は実行日の年月）。"""
    if is_target_date_set(target_date_raw):
        return format_year_month_label(target_date_raw)
    return format_year_month_label(dt.date.today())


def build_spreadsheet_title(target_date_raw) -> str:
    return f'インセンティブ計算表（{resolve_output_year_month(target_date_raw)}）'


def build_zyucyubi_filter(target_date_raw) -> str | None:
    """日付未入力なら None（全期間）、入力ありなら zyucyubi__c の年月で一致判定。"""
    if not is_target_date_set(target_date_raw):
        return None
    target_date = to_target_date(target_date_raw)
    return (
        f'CALENDAR_YEAR({ZYUCYUBI_FIELD}) = {target_date.year} '
        f'AND CALENDAR_MONTH({ZYUCYUBI_FIELD}) = {target_date.month}'
    )

def _normalize_private_key(raw_key: str) -> str:
    return str(raw_key).strip().replace('\\n', '\n')


def sf_auth():
    sf_client_id = userdata.get('SF_CLIENT_ID')
    sf_username = userdata.get('SF_USERNAME')
    sf_private_key = userdata.get('SF_PRIVATE_KEY')
    if not sf_client_id or not sf_username or not sf_private_key:
        raise RuntimeError(
            'Salesforce シークレット (SF_CLIENT_ID / SF_USERNAME / SF_PRIVATE_KEY) が見つかりません。'
        )

    now = int(time.time())
    claims = {
        'iss': sf_client_id,
        'sub': sf_username,
        'aud': 'https://login.salesforce.com',
        'exp': now + 180,
    }
    assertion = jwt.encode(
        claims,
        _normalize_private_key(sf_private_key),
        algorithm='RS256',
    )
    token_resp = requests.post(
        SF_TOKEN_URL,
        data={
            'grant_type': 'urn:ietf:params:oauth:grant-type:jwt-bearer',
            'assertion': assertion,
        },
        timeout=60,
    )
    token_resp.raise_for_status()
    token_json = token_resp.json()
    return token_json['access_token'], token_json['instance_url']


def fetch_all_soql_records(instance_url: str, access_token: str, soql: str):
    records = []
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json',
    }
    query_url = f"{instance_url}/services/data/v57.0/query?q={requests.utils.quote(soql)}"
    while True:
        resp = requests.get(query_url, headers=headers, timeout=120)
        if not resp.ok:
            try:
                print('SOQL request failed:', resp.status_code, resp.text)
            except Exception:
                pass
        resp.raise_for_status()
        payload = resp.json()
        records.extend(payload.get('records', []) or [])
        if payload.get('done'):
            break
        next_url = payload.get('nextRecordsUrl')
        if not next_url:
            break
        query_url = f"{instance_url}{next_url}"
    return records


def _to_float(value) -> float:
    try:
        return float(value or 0)
    except (TypeError, ValueError):
        return 0.0


def fetch_half_arari_totals(
    instance_url: str,
    access_token: str,
    where_clause: str | None = None,
) -> tuple[dict[str, float], dict[str, float]]:
    """担当者ごとに arari__c / 2 の合計を㊤・㊦それぞれ集計する。"""
    soql = (
        f'SELECT {UE_TANTO_FIELD}, {SHITA_TANTO_FIELD}, arari__c'
        ' FROM kado__c'
    )
    if where_clause:
        soql += f' WHERE {where_clause}'
    records = fetch_all_soql_records(instance_url, access_token, soql)

    ue_totals: dict[str, float] = {}
    shita_totals: dict[str, float] = {}
    for rec in records:
        arari_half = _to_float(rec.get('arari__c')) / 2
        ue_tanto = str(rec.get(UE_TANTO_FIELD) or '').strip()
        shita_tanto = str(rec.get(SHITA_TANTO_FIELD) or '').strip()
        if ue_tanto:
            ue_totals[ue_tanto] = ue_totals.get(ue_tanto, 0) + arari_half
        if shita_tanto:
            shita_totals[shita_tanto] = shita_totals.get(shita_tanto, 0) + arari_half
    return ue_totals, shita_totals


def employee_full_name(employee: dict) -> str:
    return f"{employee.get('last_name', '')} {employee.get('first_name', '')}".strip()


scopes = [
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/spreadsheets',
]
creds, _ = google.auth.default(scopes=scopes)
gc = gspread.authorize(creds)

employees = [
    employee for employee in get_employee_info()
    if employee.get('department') == '営業'
]

access_token, instance_url = sf_auth()
zyucyubi_filter = build_zyucyubi_filter(TARGET_DATE)
ue_arari_totals, shita_arari_totals = fetch_half_arari_totals(
    instance_url,
    access_token,
    where_clause=zyucyubi_filter,
)

if zyucyubi_filter:
    print(f'絞り込み: {format_year_month_label(TARGET_DATE)}（{ZYUCYUBI_FIELD}）')
else:
    print('絞り込み: 全期間')

rows = []
for employee in employees:
    name = employee_full_name(employee)
    ue_total = ue_arari_totals.get(name, 0)
    shita_total = shita_arari_totals.get(name, 0)
    rows.append([name, ue_total, shita_total, ue_total + shita_total])

year_month_label = resolve_output_year_month(TARGET_DATE)
spreadsheet_title = build_spreadsheet_title(TARGET_DATE)
sheet_title = year_month_label

sh = gc.create(spreadsheet_title)
ws = sh.sheet1
ws.update_title(sheet_title)

ws.update(values=[HEADER], range_name='A1:D1', value_input_option='USER_ENTERED')

if rows:
    ws.update(values=rows, range_name=f'A2:D{1 + len(rows)}', value_input_option='USER_ENTERED')

print(f'スプレッドシートを作成しました: {sh.url}')
print(f'出力件数: {len(rows)} 件')


絞り込み: 2026年03月（zyucyubi__c）
スプレッドシートを作成しました: https://docs.google.com/spreadsheets/d/10XmMWk4viZiicWchvNgZ4kj8s38NH4mJl138eeqrKZE
出力件数: 16 件
